# Free Distributed Training: SFT + DPO with LoRA

## How to Get Free Multi-GPU

| Platform | Free GPUs | How |
|----------|-----------|-----|
| **Kaggle** | **2x T4 (30GB total)** | Notebooks > New > Settings > GPU T4 x2 |
| Colab Free | 1x T4 (15GB) | Still works — ZeRO offloads to CPU |
| Colab Pro | 1x A100 (40-80GB) | Paid but powerful |

**Best free option: Kaggle 2x T4.** This notebook auto-detects your GPUs and scales.

## What This Notebook Does

```
Part 1: SFT + LoRA + DeepSpeed    (teach the model to follow instructions)
Part 2: DPO + LoRA + DeepSpeed    (align it with human preferences)
```

Both use:
- **LoRA** — train ~1% of parameters (fits on small GPUs)
- **DeepSpeed ZeRO-2** — offload optimizer to CPU (saves ~4x GPU memory)
- **Accelerate** — auto-distributes across however many GPUs you have

### Why Script Files?

Distributed training needs `accelerate launch` to spawn workers.
We write `.py` scripts and launch them — this is how real distributed training works.
Each script is ~50 lines. No magic.

---
**Runtime:** Kaggle 2x T4 (free) or Colab T4 (free)

## Step 1: Install

In [ ]:
!pip install -q transformers datasets peft accelerate "trl>=0.12" deepspeed

## Step 2: Detect GPUs

In [ ]:
import torch
import os
import json
import time

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")

print(f"\nTotal GPUs: {NUM_GPUS}")
if NUM_GPUS >= 2:
    print("Multi-GPU detected! Training will distribute across all GPUs.")
else:
    print("Single GPU — DeepSpeed will offload optimizer to CPU to save memory.")

## Step 3: Write Config Files

Two files:
1. **DeepSpeed config** — tells DeepSpeed what to offload
2. **Accelerate config** — tells Accelerate how many GPUs to use

In [ ]:
# DeepSpeed config as a standalone JSON (used by training scripts via deepspeed= arg)
ds_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True,
        },
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True,
    },
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
}

with open("ds_config.json", "w") as f:
    json.dump(ds_config, f, indent=2)

# Accelerate config — use accelerate-managed DeepSpeed (no deepspeed_config_file)
# This avoids the conflict where accelerate rejects overlapping settings
accel_yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: none
  zero3_init_flag: false
  zero_stage: 2
machine_rank: 0
main_process_ip: null
main_process_port: null
main_training_function: main
mixed_precision: bf16
num_machines: 1
num_processes: {NUM_GPUS}
use_cpu: false
"""

accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
os.makedirs(accel_dir, exist_ok=True)
with open(os.path.join(accel_dir, "default_config.yaml"), "w") as f:
    f.write(accel_yaml)

print(f"DeepSpeed: ZeRO-2 + CPU optimizer offload")
print(f"Accelerate: {NUM_GPUS} GPU(s) with DeepSpeed backend")

---
# Part 1: Distributed SFT + LoRA

**SFT (Supervised Fine-Tuning)** = teach the model to follow instructions.

We write a training script, then launch it with `accelerate launch`.
Accelerate spawns 1 worker per GPU, DeepSpeed handles memory optimization.

In [ ]:
%%writefile train_sft.py
"""Distributed SFT + LoRA training script."""
import torch, os
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True,
)

# Load a small slice of Alpaca for demo
dataset = load_dataset("tatsu-lab/alpaca", split="train")
dataset = dataset.shuffle(seed=42).select(range(500))

def format_example(ex):
    prompt = ex["instruction"] + ("\n" + ex["input"] if ex.get("input") else "")
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": ex["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print(f"SFT dataset: {len(dataset)} examples")

# Train — no deepspeed= arg; accelerate's config handles DeepSpeed
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="./sft_output",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=5,
        bf16=True,
        gradient_checkpointing=True,
        max_seq_length=512,
        dataset_text_field="text",
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM",
    ),
)

trainer.train()
trainer.save_model("./sft_output/final")
tokenizer.save_pretrained("./sft_output/final")

# Quick test
if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    model.eval()
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": "What is gravity?"}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    print("\n" + "="*50)
    print("SFT TEST — What is gravity?")
    print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    print("="*50)

In [ ]:
print(f"Launching SFT on {NUM_GPUS} GPU(s)...")
start = time.time()

!accelerate launch --num_processes={NUM_GPUS} train_sft.py

sft_time = time.time() - start
print(f"\nSFT complete! Time: {sft_time:.0f}s")

---
# Part 2: Distributed DPO + LoRA

**DPO (Direct Preference Optimization)** = align the model with human preferences.

Same pattern: write script, launch with `accelerate launch`.

DPO trains on **preference pairs** (chosen vs rejected) instead of instruction-output pairs.

In [ ]:
%%writefile train_dpo.py
"""Distributed DPO + LoRA training script."""
import torch, os
os.environ["WANDB_DISABLED"] = "true"

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True,
)

# Preference pairs
pairs = [
    {"prompt": "Explain what a black hole is.",
     "chosen": "A black hole is a region in space where gravity is so strong that nothing, not even light, can escape. They form when massive stars collapse at the end of their life cycle.",
     "rejected": "A black hole is a hole that is black. It sucks things in."},
    {"prompt": "How do I make scrambled eggs?",
     "chosen": "Crack 2-3 eggs into a bowl, whisk with salt and pepper. Heat butter in a pan over medium-low heat, pour in eggs, and gently stir until softly set.",
     "rejected": "Put eggs in pan. Cook them. Add stuff if you want."},
    {"prompt": "What is machine learning?",
     "chosen": "Machine learning is a branch of AI where computers learn patterns from data instead of being explicitly programmed. For example, a spam filter learns from labeled emails.",
     "rejected": "Machine learning is when computers learn stuff. It's really complicated."},
    {"prompt": "Why is exercise important?",
     "chosen": "Regular exercise strengthens your heart, improves mood by releasing endorphins, helps maintain healthy weight, and reduces chronic disease risk.",
     "rejected": "Exercise is good for you. You should do it because everyone says so."},
    {"prompt": "Explain recursion in programming.",
     "chosen": "Recursion is when a function calls itself to solve smaller sub-problems. For example, factorial(5) = 5 * factorial(4). Every recursive function needs a base case to stop.",
     "rejected": "Recursion is a hard concept. It's when things repeat."},
    {"prompt": "What is photosynthesis?",
     "chosen": "Photosynthesis is how plants convert sunlight, water, and CO2 into glucose and oxygen. It happens in chloroplasts using chlorophyll.",
     "rejected": "Plants eat sunlight somehow. It's a biology thing."},
    {"prompt": "How does the internet work?",
     "chosen": "The internet is a global network of computers. When you visit a website, your browser sends a request to a server via HTTP/TCP, and it sends back the page data.",
     "rejected": "The internet is wifi. You connect and it works."},
    {"prompt": "What is the theory of relativity?",
     "chosen": "Einstein's relativity: special relativity says light speed is constant and time slows at high speeds. General relativity says massive objects curve spacetime, causing gravity.",
     "rejected": "Relativity is Einstein's thing about E=mc2. Everything is relative."},
]

def fmt(p, r):
    return [{"role": "user", "content": p}, {"role": "assistant", "content": r}]

rows = [{"prompt": [{"role": "user", "content": e["prompt"]}],
         "chosen": fmt(e["prompt"], e["chosen"]),
         "rejected": fmt(e["prompt"], e["rejected"])} for e in pairs]
dataset = Dataset.from_list(rows)
print(f"DPO dataset: {len(dataset)} preference pairs")

# Train — no deepspeed= arg; accelerate's config handles DeepSpeed
trainer = DPOTrainer(
    model=model,
    args=DPOConfig(
        output_dir="./dpo_output",
        beta=0.1,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        warmup_steps=5,
        logging_steps=1,
        bf16=True,
        gradient_checkpointing=True,
        do_eval=False,
        remove_unused_columns=False,
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none", task_type="CAUSAL_LM",
    ),
)

trainer.train()
trainer.save_model("./dpo_output/final")
tokenizer.save_pretrained("./dpo_output/final")

# Quick test
if int(os.environ.get("LOCAL_RANK", 0)) == 0:
    model.eval()
    test_prompts = ["What is gravity?", "How do I learn Python?", "What makes a good friend?"]
    for p in test_prompts:
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": p}],
            tokenize=False, add_generation_prompt=True,
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
        resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        print(f"\nQ: {p}")
        print(f"A: {resp[:200]}")

In [ ]:
print(f"Launching DPO on {NUM_GPUS} GPU(s)...")
start = time.time()

!accelerate launch --num_processes={NUM_GPUS} train_dpo.py

dpo_time = time.time() - start
print(f"\nDPO complete! Time: {dpo_time:.0f}s")

---
# Results

In [ ]:
from IPython.display import HTML, display

html = f"""
<div style="font-family:sans-serif; max-width:700px;">
<h2 style="color:#a78bfa;">Distributed Training Results</h2>
<p style="color:#888;">GPUs: {NUM_GPUS} x {torch.cuda.get_device_name(0)}</p>
<table style="width:100%; border-collapse:collapse; background:#131927; color:#ccc;">
<tr style="border-bottom:2px solid #4f46e5;">
  <th style="text-align:left; padding:12px; color:#a78bfa;">Method</th>
  <th style="text-align:left; padding:12px; color:#a78bfa;">Time</th>
  <th style="text-align:left; padding:12px; color:#a78bfa;">What It Does</th>
</tr>
<tr style="border-bottom:1px solid #333;">
  <td style="padding:12px; color:#34d399; font-weight:bold;">SFT + LoRA</td>
  <td style="padding:12px;">{sft_time:.0f}s</td>
  <td style="padding:12px;">Teach model to follow instructions (500 Alpaca examples)</td>
</tr>
<tr>
  <td style="padding:12px; color:#f87171; font-weight:bold;">DPO + LoRA</td>
  <td style="padding:12px;">{dpo_time:.0f}s</td>
  <td style="padding:12px;">Align with preferences (8 chosen/rejected pairs)</td>
</tr>
</table>
<p style="color:#888; margin-top:15px;">Both used: DeepSpeed ZeRO-2 + CPU offload + LoRA + Accelerate launcher</p>
</div>
"""
display(HTML(html))

---
## How It All Fits Together

```
You write:         train_sft.py  or  train_dpo.py
                         |
                         v
accelerate launch --num_processes=N  train_xxx.py
                         |
              +----------+----------+
              |          |          |
           Worker 0   Worker 1   Worker N
           (GPU 0)    (GPU 1)    (GPU N)
              |          |          |
              +--- DeepSpeed ZeRO-2 ---+
              |  shards optimizer+grads |
              +--- across all GPUs -----+
              |          |          |
              +--- LoRA adapters ---+
                  (tiny, on GPU)
```

### What Each Piece Does

| Tool | Job | One-liner |
|------|-----|-----------|
| **Accelerate** | Launcher + Config | Spawns N processes, configures DeepSpeed |
| **DeepSpeed** | Memory | Offloads optimizer to CPU, shards gradients |
| **LoRA** | Efficiency | Only trains ~1% of model parameters |
| **SFTTrainer** | SFT | Trains on instruction-output pairs |
| **DPOTrainer** | DPO | Trains on chosen-vs-rejected preference pairs |

### The Only Distributed Code

There's literally nothing special in the scripts. The entire "distributed" part is:

1. The accelerate config YAML (written once, sets ZeRO-2 + CPU offload)
2. `accelerate launch --num_processes=N` to run it

That's it. The training code is identical to single-GPU code.

### Free GPU Options

| Platform | GPUs | Cost | Best For |
|----------|------|------|----------|
| **Kaggle** | 2x T4 (30GB) | Free (30h/week) | Multi-GPU experiments |
| **Colab** | 1x T4 (15GB) | Free | Quick tests |
| **Lightning.ai** | 1x T4/A10 | Free tier | Persistent sessions |
| **Paperspace** | 1x M4000 (8GB) | Free tier | Always available |